In [ ]:
from qiskit import QuantumCircuit
 
# Create a new circuit with two qubits
qc = QuantumCircuit(2)
 
# Add a Hadamard gate to qubit 0
qc.h(0)
 
# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)
 
# Return a text drawing of the circuit.
qc.draw()

In [ ]:
from qiskit import QuantumCircuit
 
# Create a new circuit with two qubits
qc = QuantumCircuit(2)
 
qc.h(0)
 
qc.x(0)

qc.y(0)

qc.z(0)

qc.draw()

In [ ]:
from qiskit import QuantumCircuit
import math

# 1. Create a circuit with 2 qubits
qc = QuantumCircuit(2)

# Define our parameters
c, t = 0, 1  # Control = Qubit 0, Target = Qubit 1
a, b = 0, 1  # For swapping
theta = math.pi / 2  # 90 degree phase shift

# 2. Add the gates you requested
qc.cx(c, t)         # Controlled-X (CNOT)
qc.cz(c, t)         # Controlled-Z
qc.barrier()        # Visual separator

qc.swap(a, b)       # Swap Qubit 0 and 1
qc.cp(theta, c, t)  # Controlled-Phase
qc.iswap(a, b)      # iSwap

# 3. Draw the circuit
# Note: 'mpl' makes it look like a textbook diagram; 'text' works in the console.
print(qc.draw(output='text'))

In [ ]:
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

# 1. Create the circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.x(2)
qc.barrier()
qc.ccx(0, 1, 2)
qc.measure_all()

# 2. Draw the circuit using Matplotlib
# In a Jupyter notebook, this line alone will show the image.
# In a standard Python script, you need to capture the figure and show it.
qc.draw(output='mpl')

In [ ]:
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

# --- 1) Build a Bell-state circuit (|Φ+> = (|00> + |11>)/√2) ---
qc = QuantumCircuit(2, 2)
qc.h(0)          # Put qubit 0 in superposition
qc.cx(0, 1)      # Entangle: control q0 -> target q1
qc.barrier()
qc.measure([0, 1], [0, 1])

# Draw the circuit
circuit = qc.draw('mpl')
plt.show()

In [ ]:
code = """
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

qc = QuantumCircuit(2, 2)
qc.h(0)         
qc.cx(0, 1)      
qc.barrier()
qc.measure([0, 1], [0, 1])

# Draw the circuit
circuit = qc.draw('mpl')
plt.show()
"""

exec_globals = {}

exec(code, exec_globals)

exec_globals

In [ ]:
def move_imports(code: str) -> str:
    """
    Move all import statements to the top of the code string.
    """
    lines = code.splitlines()
    import_lines = []
    other_lines = []

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("import") or stripped.startswith("from"):
            import_lines.append(line)
        else:
            other_lines.append(line)

    return "\n".join(import_lines + [""] + other_lines)


# Example usage
code = """
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.barrier()
qc.measure([0, 1], [0, 1])

import matplotlib.pyplot as plt

circuit = qc.draw("mpl")
plt.show()
"""

fixed_code = move_imports(code)
print(fixed_code)


In [ ]:
import os
from rope.base.project import Project
from rope.refactor.importutils import ImportOrganizer

# Create a project directory
project_root = '/tmp/rope_project'
os.makedirs(project_root, exist_ok=True)

file_path = os.path.join(project_root, 'temp.py')

code_str = """
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.barrier()
qc.measure([0, 1], [0, 1])
import matplotlib.pyplot as plt

circuit = qc.draw('mpl')
plt.show()
"""

# Write the code string to disk
with open(file_path, 'w') as f:
    f.write(code_str)

# Create Rope project and resource
project = Project(project_root)
resource = project.get_file('temp.py')

# Correct usage: only pass project to ImportOrganizer
organizer = ImportOrganizer(project)

# Organize imports for the resource
changes = organizer.organize_imports(resource)
project.do(changes)

# Read back the refactored code
with open(file_path) as f:
    new_code = f.read()

print(new_code)


In [ ]:
import tempfile
import os
from rope.base.project import Project
from rope.refactor.importutils import ImportOrganizer

def fix_imports(code_str: str) -> str:
    """
    Takes Python code as a string, uses Rope to organize imports,
    and returns the refactored code string.
    All work is done in a temporary directory.
    """
    # Create a temporary directory for the Rope project
    with tempfile.TemporaryDirectory(prefix="rope_project_") as project_root:
        file_path = os.path.join(project_root, "temp.py")

        # Write the code string to a temporary file
        with open(file_path, "w") as f:
            f.write(code_str)

        # Create Rope project and resource
        project = Project(project_root)
        resource = project.get_file("temp.py")

        # Run ImportOrganizer
        organizer = ImportOrganizer(project)
        changes = organizer.organize_imports(resource)
        project.do(changes)

        # Read back the refactored code
        with open(file_path) as f:
            new_code = f.read()

        project.close()

    return new_code

code = """
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.barrier()
qc.measure([0, 1], [0, 1])
import matplotlib.pyplot as plt

circuit = qc.draw('mpl')
plt.show()
"""

print(fix_imports(code))


In [ ]:
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt

# --- 1) Build a Bell-state circuit (|Φ+> = (|00> + |11>)/√2) ---
qc = QuantumCircuit(2, 2)
qc.h(0)          # Put qubit 0 in superposition
qc.cx(0, 1)      # Entangle: control q0 -> target q1
qc.barrier()
qc.measure([0, 1], [0, 1])

# Draw the circuit
display(qc.draw('mpl'))

# --- 2) Transpile for AerSimulator and run shot-based simulation ---
backend = AerSimulator()
tqc = transpile(qc, backend)   # <- required step to match backend
job = backend.run(tqc, shots=1024)
result = job.result()
counts = result.get_counts()

# Display the measurement counts result
print("Measurement counts:", counts)

# Plot the histogram the circuit result
plot_histogram(counts)

In [ ]:
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt

# --- 1) Build a Bell-state circuit (|Φ+> = (|00> + |11>)/√2) ---
qc = QuantumCircuit(2, 2)
qc.h(0)          # Put qubit 0 in superposition
qc.cx(0, 1)      # Entangle: control q0 -> target q1
qc.barrier()
qc.measure([0, 1], [0, 1])

# Draw the circuit
display(qc.draw('mpl'))

# --- 2) Transpile for AerSimulator and run shot-based simulation ---
backend = AerSimulator()
tqc = transpile(qc, backend)
job = backend.run(tqc, shots=1024)
result = job.result()
counts = result.get_counts()

print("Measurement counts:", counts)

# Plot the histogram of the circuit result
plot_histogram(counts)
plt.show()

# --- 3) Display Bloch sphere for the statevector before measurement ---
# Build the circuit again but without measurement to get the pure state
qc_state = QuantumCircuit(2)
qc_state.h(0)
qc_state.cx(0, 1)

# Get the statevector
state = Statevector.from_instruction(qc_state)

# Plot Bloch sphere representation
fig = plot_bloch_multivector(state)
fig.set_size_inches(6, 6)  # resize if needed
plt.show()


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_state_qsphere
 
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
 
state = Statevector(qc)
plot_state_qsphere(state)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
 
qc = QuantumCircuit(2)
qc.h(0)
qc.x(1)
 
state = Statevector(qc)
plot_bloch_multivector(state)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
 
qc = QuantumCircuit(2)
qc.h(0)
qc.x(1)
 
state = Statevector(qc)
plot_bloch_multivector(state)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
import streamlit as st

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

# Get the Matplotlib figure
fig = circuit_drawer(qc, output="mpl", style={"backgroundcolor": "#EEEEEE"})

# Show it later in Streamlit
st.pyplot(fig)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
import matplotlib.pyplot as plt

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

# Get the Matplotlib figure
fig = circuit_drawer(qc, output="mpl", scale=0.5)

plt.show(fig)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import streamlit as st

# Build a simple circuit
qc = QuantumCircuit(1, 1)
state = Statevector(qc)

# Get the Bloch sphere figure
fig = plot_bloch_multivector(state)

plt.show()


In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
import matplotlib.pyplot as plt

qc = QuantumCircuit(5, 1)
qc.h(0)
qc.measure(0, 0)

# Get the Matplotlib figure
circuit_drawer(qc, output="mpl", scale=0.1)



In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import streamlit as st

# Build a simple circuit
qc = QuantumCircuit(4, 1)
state = Statevector(qc)

# Get the Bloch sphere figure
plot_bloch_multivector(state, figsize=(3, 3))



In [ ]:
# To run this code you need to install the following dependencies:
# pip install google-genai

import base64
import os
from google import genai
from google.genai import types


def generate():
    client = genai.Client(
        api_key="",
    )

    model = "gemini-robotics-er-1.5-preview"
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text="""hello"""),
            ],
        ),
    ]
    tools = [
        types.Tool(googleSearch=types.GoogleSearch(
        )),
    ]
    generate_content_config = types.GenerateContentConfig(
        tools=tools,
    )

    for chunk in client.models.generate_content_stream(
        model=model,
        contents=contents,
        config=generate_content_config,
    ):
        print(chunk.text, end="")

if __name__ == "__main__":
    generate()


Hello! How can I help you today?